# SD1.5 : première baseline DDIM sur Colab T4

**Périmètre : environnement + BASE uniquement.** Aucun reward, COV, FULL ou LiDAR.

Paramètres : SD1.5, 512 × 512, DDIM 50 pas, $\eta=0$, CFG 7.5, batch 1.
Premier pilote : les **5 premiers prompts dev**, présents dans `generation_prompts.txt`, seed 0.
Le jeu heldout n'est pas utilisé pour générer ni régler quoi que ce soit.

Le code réutilisable est dans `baseline.py`, la validation dans `validate_baseline.py`.
Ce notebook appelle ces fichiers et en affiche les fonctions importantes : pas de seconde
implémentation à maintenir entre Colab et Northflank.

**Statut à la livraison :** voir `validation/` pour les contrôles locaux effectués.
La validation du vrai SD1.5 sur GPU doit être exécutée ici. Les cellules n'ont pas de résultats préremplis.

## Mode d'emploi

1. Dans Colab, choisir **Modifier → Paramètres du notebook → GPU T4**.
2. Exécuter la première cellule et importer `diffusion-reward-guidance.zip` lorsqu'elle le demande.
3. Installer les versions figées. Si demandé, redémarrer la session et reprendre à la cellule de repérage du projet.
4. Exécuter les cellules dans l'ordre. Un écart de validation doit arrêter le pilote.
5. Télécharger le ZIP de résultats à la fin, ou utiliser le dossier Drive optionnel.

La première installation peut prendre plusieurs minutes et télécharge les poids du modèle au chargement.


In [ ]:
from pathlib import Path
import os
import sys
import zipfile

# Cette cellule peut être relancée après un redémarrage du runtime.
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/diffusion-reward-guidance')]
PROJECT_DIR = next((p for p in candidates if (p / 'baseline.py').is_file()), None)
if PROJECT_DIR is None:
    from google.colab import files
    print('Importer diffusion-reward-guidance.zip')
    uploaded = files.upload()
    archive_name = next(name for name in uploaded if name.endswith('.zip'))
    destination = Path('/content').resolve()
    with zipfile.ZipFile(archive_name) as archive:
        # Refuser les chemins sortant du dossier de destination.
        for entry in archive.infolist():
            (destination / entry.filename).resolve().relative_to(destination)
        archive.extractall(destination)
    PROJECT_DIR = destination / 'diffusion-reward-guidance'
assert (PROJECT_DIR / 'baseline.py').is_file(), 'Dossier diffusion-reward-guidance introuvable.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Projet :', PROJECT_DIR)


## 1. Versions reproductibles

Base choisie : **PyTorch 2.8.0, CUDA 12.6, Diffusers 0.35.1, Transformers 4.51.3, Tokenizers 0.21.4**.
Les versions applicatives sont dans `requirements.txt`. La révision Hugging Face des poids est figée dans le code.

Colab contient déjà souvent PyTorch, torchvision et torchaudio : les trois sont alignés pour éviter
des extensions binaires incompatibles. Le projet n'utilise pas torchaudio. Sur Northflank, les mêmes
versions ciblent Python 3.10-3.13 avec un pilote compatible CUDA 12.6.
Tokenizers 0.21.4 fournit une wheel compatible Python 3.13, contrairement à l'ancienne version 0.19.1.
Les tests locaux utilisent Python 3.12 ; la validation Python 3.13 / T4 se fait dans ce notebook.
Le driver et l'environnement réellement utilisés seront enregistrés avec les résultats.

Cette cellule installe avant d'importer torch. **Après une modification des paquets, redémarrer la session** :
les bibliothèques déjà chargées par Colab ne sont pas remplacées en mémoire par pip.
Lors de la première installation, exécuter uniquement les deux premières cellules de code, puis redémarrer.
Les sorties de pip sont affichées et sauvegardées dans `installation_logs/`.


In [ ]:
import importlib.metadata
import subprocess

assert (3, 10) <= sys.version_info[:2] <= (3, 13), 'Utiliser Python 3.10 à 3.13.'
print('Python :', sys.version, flush=True)

# Mettre à jour une archive de la première livraison, déjà importée dans Colab.
requirements_path = PROJECT_DIR / 'requirements.txt'
requirements_text = requirements_path.read_text(encoding='utf-8')
if 'transformers==4.44.2' in requirements_text:
    requirements_text = requirements_text.replace(
        'transformers==4.44.2', 'transformers==4.51.3\ntokenizers==0.21.4',
    )
    requirements_path.write_text(requirements_text, encoding='utf-8')
    print('Ancien requirements.txt corrigé pour Python 3.13.', flush=True)

# requirements.txt reste la référence unique des versions applicatives.
expected = dict(
    line.strip().split('==') for line in requirements_text.splitlines()
    if line.strip() and not line.lstrip().startswith('#')
)
torch_expected = {
    'torch': '2.8.0+cu126', 'torchvision': '0.23.0+cu126', 'torchaudio': '2.8.0+cu126',
}
expected.update(torch_expected)
installed = {}
for package in expected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
versions_mismatch = any(installed[p] != v for p, v in expected.items())
# Relancer cette cellule sans redémarrer ne doit pas effacer cet avertissement.
RESTART_REQUIRED = globals().get('RESTART_REQUIRED', False) or versions_mismatch

def run_visible(command, log_name):
    # Afficher stdout/stderr du sous-processus dans Colab et conserver le journal.
    log_dir = PROJECT_DIR / 'installation_logs'
    log_dir.mkdir(exist_ok=True)
    with (log_dir / log_name).open('w', encoding='utf-8') as log:
        with subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Échec (code {return_code}). Voir les lignes ci-dessus et {log_dir / log_name}.')

if versions_mismatch:
    if any(installed[p] != v for p, v in torch_expected.items()):
        run_visible([
            sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off',
            '--only-binary=:all:',
            'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
            '--index-url', 'https://download.pytorch.org/whl/cu126',
        ], 'pytorch.log')
    run_visible([
        sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off',
        '--only-binary=:all:', '-r', str(requirements_path),
    ], 'requirements.log')
    print('STOP : installation terminée. Redémarrer la session, puis reprendre à la première cellule.')
elif RESTART_REQUIRED:
    print('STOP : les paquets sont installés, mais la session doit encore être redémarrée.')
else:
    print('Les versions attendues sont déjà installées.')


In [ ]:
assert not globals().get('RESTART_REQUIRED', False), 'Redémarrer la session après installation.'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import diffusers
import transformers

assert torch.cuda.is_available(), 'Sélectionner un runtime GPU T4 dans les paramètres Colab.'
assert torch.__version__.split('+')[0] == '2.8.0'
assert diffusers.__version__ == '0.35.1'
assert transformers.__version__ == '4.51.3'
assert importlib.metadata.version('tokenizers') == '0.21.4'
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), 'GiB')
print('PyTorch / CUDA :', torch.__version__, torch.version.cuda)
subprocess.run(['nvidia-smi'], check=True)


## 2. Contrôles CPU sans téléchargement de poids

Ces tests vérifient les 50 pas DDIM isolés en fp32 et fp16, les conventions de clipping et du dernier pas,
le bruit reproductible et une petite pipeline Diffusers aléatoire.
**Ils ne prouvent pas que SD1.5 fonctionne sur T4** : la validation GPU suit plus bas.


In [ ]:
run_visible([sys.executable, '-m', 'unittest', 'test_baseline', '-v'], 'cpu_tests.log')


## 3. Charger les composants de Stable Diffusion 1.5

`load_pipeline` charge tokenizer, encodeur de texte CLIP, U-Net et VAE. Les paramètres sont gelés et les
modèles passent en mode évaluation. Cette baseline utilise fp16 pour les réseaux sur T4 comme sur A100.
Le scheduler DDIM est construit **depuis la configuration SD1.5**, pas depuis les valeurs génériques de DDIM.

On compare les images brutes issues du VAE, sans filtre de sortie : les deux chemins ont le même post-traitement.
Les options mémoire qui changeraient les opérations (tiling, offload, compilation) restent désactivées pour
établir cette première équivalence.


In [ ]:
import inspect
from baseline import (
    MODEL_ID, MODEL_REVISION, configure_torch, load_pipeline, encode_prompt_cfg,
    make_initial_noise, ddim_step, sample_ddim, decode_latents, sample_official,
)
from validate_baseline import select_dev_prompts, save_environment, validate_prompt

configure_torch()
print(inspect.getsource(load_pipeline))
print('Modèle :', MODEL_ID, '\nRévision :', MODEL_REVISION)
pipe = load_pipeline()


In [ ]:
pipe.scheduler.set_timesteps(50, device=pipe._execution_device)
config = pipe.scheduler.config
assert config.num_train_timesteps == 1000
assert config.beta_schedule == 'scaled_linear'
assert config.prediction_type == 'epsilon'
assert config.timestep_spacing == 'leading'
assert config.steps_offset == 1
assert config.set_alpha_to_one is False
assert config.clip_sample is False and config.thresholding is False
assert pipe.vae_scale_factor == 8 and pipe.unet.config.in_channels == 4
assert len(pipe.scheduler.timesteps) == 50
assert pipe.scheduler.timesteps.tolist() == list(range(981, 0, -20))
assert torch.equal(pipe.scheduler.final_alpha_cumprod, pipe.scheduler.alphas_cumprod[0])
print('Timesteps :', pipe.scheduler.timesteps.tolist())
print('Alpha cumulé final :', float(pipe.scheduler.final_alpha_cumprod))
print('Échelle des latents VAE :', pipe.vae.config.scaling_factor)


## 4. Paramètres du pilote et sauvegarde

Les cinq premiers prompts dev sont : sandwich ; frisbee et téléphone ; deux bananes ; horloge verte ;
zèbre à droite d'un lit. Leur provenance est dans `data/pilot_manifest.json`.

`PROMPT_COUNT=5` et `SEED=0` sont les paramètres du premier test. Après validation, passer à 10 ou 25
ne change que la taille du préfixe dev. Le résultat de chaque prompt est écrit immédiatement.


In [ ]:
from datetime import datetime, timezone
import json

PROMPT_COUNT = 5
SEED = 0
prompts = select_dev_prompts(PROMPT_COUNT)
RUN_NAME = 'baseline_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
OUTPUT_DIR = PROJECT_DIR / 'runs' / RUN_NAME

# Option : décommenter ces lignes pour écrire directement sur Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = Path('/content/drive/MyDrive/diffusion-reward-guidance') / RUN_NAME

environment = save_environment(pipe, OUTPUT_DIR)
print('Sorties :', OUTPUT_DIR)
for index, prompt in enumerate(prompts):
    print(index, prompt)


## 5. Les équations dans le code

L'encodeur CLIP transforme séparément le prompt et le prompt vide. Les embeddings sont concaténés dans l'ordre
**inconditionnel, conditionnel**. Le U-Net prédit les deux bruits dans un batch de taille deux, puis :

$$\epsilon_{CFG}=\epsilon_u+7.5(\epsilon_c-\epsilon_u).$$

La variable `initial_noise` est un tirage gaussien brut, avant multiplication par `init_noise_sigma`.
Un générateur CPU explicite fournit une entrée réutilisable ; il ne garantit pas des sorties bit à bit identiques
sur deux architectures GPU différentes.


In [ ]:
print(inspect.getsource(encode_prompt_cfg))
print(inspect.getsource(make_initial_noise))
positive, negative = encode_prompt_cfg(pipe, prompts[0])
initial_noise = make_initial_noise(pipe, SEED)
print('Embeddings positif / négatif :', positive.shape, negative.shape)
print('Latents initiaux :', initial_noise.shape, initial_noise.dtype)


La boucle DDIM calcule, pour chaque timestep $t$, avec $s=t-1000/50$ :

$$\widehat z_0 = \frac{z_t-\sqrt{1-\bar\alpha_t}\,\epsilon_{CFG}}{\sqrt{\bar\alpha_t}},$$
$$z_s = \sqrt{\bar\alpha_s}\,\widehat z_0 + \sqrt{1-\bar\alpha_s}\,\epsilon_{CFG}.$$

Dans la notation du rapport, $\alpha_t=\sqrt{\bar\alpha_t}$ et $\sigma_t=\sqrt{1-\bar\alpha_t}$.
**Dernier pas :** si $s<0$, Diffusers emploie `final_alpha_cumprod`. Pour SD1.5, ce coefficient vaut
`alphas_cumprod[0]`, pas un. Les coefficients restent des scalaires fp32 comme dans le scheduler officiel.
Le résultat tensoriel suit l'arithmétique fp16 de la référence ; aucun cast supplémentaire ne change la comparaison.

La fonction suivante implémente directement ce pas. La boucle utilise le U-Net et cette fonction, sans appeler
`scheduler.step`. Les états sont explicitement détachés entre les étapes. Cette version BASE est sous `no_grad` ;
une future méthode FULL exigera d'activer le gradient pour le pas courant.


In [ ]:
print(inspect.getsource(ddim_step))
print(inspect.getsource(sample_ddim))
print(inspect.getsource(decode_latents))


## 6. Validation GPU contre la pipeline officielle

Pour chaque prompt, `validate_prompt` :

1. Compare nos embeddings à `pipe.encode_prompt`.
2. Donne **les mêmes tensors** de bruit et d'embeddings aux deux chemins.
3. Compare la trajectoire à chacun des 50 timesteps, les latents finaux puis les images flottantes.
4. Vérifie formes, valeurs finies, configuration et répétabilité (premier prompt).
5. Écrit un rapport JSON, les tenseurs et deux PNG ; arrête l'exécution en cas de désaccord.

L'égalité exacte est attendue à mêmes opérations. Une marge **atol=rtol=1e-5**, fixée avant les runs, est
également vérifiée et documentée : elle est inférieure à un ulp fp16 autour de 1. Les statistiques exactes
sont conservées. Une divergence ne doit pas être masquée en augmentant la tolérance.

Les temps incluent génération, décodage, traces CPU et diagnostics. Ils excluent le téléchargement,
le chargement des poids et l'encodage du texte. Les pics indiquent mémoire totale et surplus par rapport
aux poids déjà chargés. Ce sont des mesures de validation, pas encore un benchmark de débit optimisé.


In [ ]:
# Chauffe du U-Net et du VAE, exclue des mesures. Les runs validés auront 50 pas.
warm_latents, _ = sample_ddim(pipe, initial_noise, positive, negative, num_steps=1)
warm_image = decode_latents(pipe, warm_latents)
del warm_latents, warm_image
print(inspect.getsource(sample_official))


In [ ]:
reports = []
for index, prompt in enumerate(prompts):
    print(f'[{index + 1}/{len(prompts)}] {prompt}', flush=True)
    report = validate_prompt(
        pipe, prompt, SEED, OUTPUT_DIR / f'prompt_{index:02d}', repeat=(index == 0),
    )
    reports.append(report)
    print('PASS | égalité exacte latente :', report['final_latents']['exact_equal'])
    print('Erreur max latente :', report['final_latents']['max_absolute_error'])
    print('Erreur max image :', report['images_before_png']['max_absolute_error'])
    print('Coût officiel :', report['official_cost'])
    print('Coût notre boucle :', report['custom_cost'])

(OUTPUT_DIR / 'summary.json').write_text(json.dumps(reports, indent=2), encoding='utf-8')
assert len(reports) == PROMPT_COUNT and all(r['passed'] for r in reports)
print('Validation GPU terminée pour ces prompts, cette machine et cet environnement.')


## 7. Images et résultats à conserver

À gauche : Diffusers ; à droite : notre boucle. La comparaison numérique porte déjà sur les images avant PNG.
Les latents initiaux et finaux sont dans `tensors.pt`, le premier pas divergent éventuel dans `validation.json`.
Si une cellule a échoué, consulter ce fichier et arrêter l'extension à davantage de prompts.


In [ ]:
from PIL import Image
from IPython.display import display

for index, prompt in enumerate(prompts):
    official_image = Image.open(OUTPUT_DIR / f'prompt_{index:02d}' / 'official.png')
    custom_image = Image.open(OUTPUT_DIR / f'prompt_{index:02d}' / 'custom.png')
    comparison = Image.new('RGB', (1024, 512))
    comparison.paste(official_image, (0, 0))
    comparison.paste(custom_image, (512, 0))
    comparison.save(OUTPUT_DIR / f'comparison_{index:02d}.png')
    print(prompt)
    display(comparison)


In [ ]:
import shutil

result_zip = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Archive des résultats :', result_zip)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(result_zip)


## Après ce notebook

Ne commencer COV, FULL et LiDAR qu'après examen de cette validation. Pour reprendre sur Northflank A100,
déployer le même dossier, installer les mêmes versions et lancer :

```bash
python validate_baseline.py --count 5 --seed 0 --output runs/a100_baseline
```

La comparaison se fait **sur chaque machine** ; les écarts T4/A100 sont distincts de l'équivalence de deux
implémentations sur un même GPU. Chaque run sauvegarde son environnement complet.

Sources : [rapport de recherche](../docs/report.pdf),
[DDIMScheduler 0.35.1](https://github.com/huggingface/diffusers/blob/v0.35.1/src/diffusers/schedulers/scheduling_ddim.py),
[pipeline SD 0.35.1](https://github.com/huggingface/diffusers/blob/v0.35.1/src/diffusers/pipelines/stable_diffusion/pipeline_stable_diffusion.py),
[configuration SD1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/blob/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/scheduler/scheduler_config.json).
